# Event-Level ParT Classifier — Evaluation

Evaluate the trained event-level ParticleTransformer binary classifier (HH→4b vs QCD).

Sections:
1. Load model checkpoint from W&B
2. Run inference on validation set
3. ROC curve (signal efficiency vs QCD rejection)
4. Score distributions
5. HT sculpting check
6. Baseline comparison (≥3 b-tagged jets)

In [ ]:
import json
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, auc
from torch.utils.data import DataLoader

from parT import ParticleTransformer
from parT_helpers import get_model_ckpt
from train_part_data import StratifiedJetDataset, stratified_split, precollated_collate

## 1. Load Model

In [ ]:
CONFIG_PATH = "config_event.json"

with open(CONFIG_PATH) as f:
    cfg = json.load(f)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

# Load checkpoint from W&B
wandb_path = f"{cfg['wandb']['entity']}/{cfg['wandb']['project']}"
ckpt = get_model_ckpt(
    wandb_path=wandb_path,
    artifact_name=cfg["wandb"]["artifact_name"],
    ckpt_type="best",
)

model = ParticleTransformer(
    input_dim=cfg["input_dim"],
    embed_dim=cfg["model"]["embed_dim"],
    num_pairwise_feat=cfg["model"]["num_pairwise_feat"],
    num_heads=cfg["model"]["num_heads"],
    num_layers=cfg["model"]["num_layers"],
    num_cls_layers=cfg["model"]["num_cls_layers"],
    dropout=cfg["model"]["dropout"],
    num_classes=cfg["model"]["num_classes"],
    use_batch_norm=cfg["model"].get("use_batch_norm", True),
    pt_regression=False,
    quantile_regression=False,
).to(device)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded model from epoch {ckpt['epoch']+1}, val AUC={ckpt['val_auc']:.4f}")

## 2. Inference on Validation Set

In [ ]:
dataset = StratifiedJetDataset(cfg["training"]["data"]["data_path"])
_, val_ds, _, val_indices, _ = stratified_split(
    dataset, cfg["training"]["data"]["val_split"], num_classes=1
)

val_loader = DataLoader(
    val_ds,
    batch_size=cfg["training"]["batch_size"] * 2,
    shuffle=False,
    num_workers=0,
    collate_fn=precollated_collate,
)

all_scores, all_labels, all_qcd_w, all_ht = [], [], [], []

with torch.no_grad():
    for X_batch, y_batch, mask_batch, _, jet_pt, _, _, qcd_weights in val_loader:
        X_batch = X_batch.to(device)
        mask_batch = mask_batch.to(device)

        outputs = model(X_batch, particle_mask=mask_batch)
        cls_output = outputs["classification"] if isinstance(outputs, dict) else outputs
        scores = torch.sigmoid(cls_output).cpu().numpy()

        all_scores.append(scores)
        all_labels.append(y_batch.numpy())
        all_qcd_w.append(qcd_weights.numpy())
        all_ht.append(jet_pt.numpy())  # jet_pt stores HT

scores = np.concatenate(all_scores).ravel()
labels = np.concatenate(all_labels).ravel()
qcd_w = np.concatenate(all_qcd_w).ravel()
ht = np.concatenate(all_ht).ravel()

print(f"Inference done: {len(scores)} events")
print(f"  Signal: {(labels==1).sum()}, Background: {(labels==0).sum()}")

## 3. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(labels, scores, sample_weight=qcd_w)
auc_val = roc_auc_score(labels, scores, sample_weight=qcd_w)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Standard ROC
ax1.plot(fpr, tpr, label=f"Event ParT (AUC={auc_val:.4f})")
ax1.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax1.set_xlabel("False Positive Rate (QCD efficiency)")
ax1.set_ylabel("True Positive Rate (Signal efficiency)")
ax1.set_title("ROC Curve")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Signal eff vs QCD rejection
nonzero = fpr > 0
ax2.plot(tpr[nonzero], 1.0 / fpr[nonzero], label=f"Event ParT (AUC={auc_val:.4f})")
ax2.set_xlabel("Signal Efficiency")
ax2.set_ylabel("QCD Rejection (1/FPR)")
ax2.set_yscale("log")
ax2.set_title("Signal Efficiency vs QCD Rejection")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("event_roc.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"AUC = {auc_val:.4f}")

## 4. Score Distributions

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

bins = np.linspace(0, 1, 51)
ax.hist(scores[labels == 1], bins=bins, alpha=0.6, label="HH→4b (signal)",
        density=True, histtype="stepfilled", color="tab:blue")
ax.hist(scores[labels == 0], bins=bins, alpha=0.6, label="QCD (background)",
        density=True, histtype="stepfilled", color="tab:orange",
        weights=qcd_w[labels == 0])

ax.set_xlabel("Classifier Score")
ax.set_ylabel("Normalised Density")
ax.set_title("Event-Level ParT Score Distributions")
ax.legend()
ax.set_yscale("log")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("event_scores.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. HT Sculpting Check

Verify the classifier score is not strongly correlated with event HT for QCD.

In [ ]:
bkg_mask = labels == 0

fig, ax = plt.subplots(figsize=(8, 6))
h = ax.hist2d(
    ht[bkg_mask], scores[bkg_mask],
    bins=[50, 50],
    range=[[0, np.percentile(ht[bkg_mask], 99)], [0, 1]],
    cmap="viridis",
    weights=qcd_w[bkg_mask],
)
plt.colorbar(h[3], ax=ax, label="Weighted counts")
ax.set_xlabel("Event HT (GeV)")
ax.set_ylabel("Classifier Score")
ax.set_title("QCD: Score vs HT (sculpting check)")

plt.tight_layout()
plt.savefig("event_ht_sculpting.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Baseline: b-tag Cut Comparison

Compare the ParT classifier to a simple "≥3 b-tagged jets" cut.

In [ ]:
# The baseline is effectively the trigger requirement itself.
# If we ran with trigger, all events already pass ≥3 b-tags.
# For a meaningful comparison, compute a score based on the number of
# b-tagged jets (from the dataset metadata if available) or just report
# the ParT AUC against the trigger efficiency.

print(f"Event-level ParT AUC: {auc_val:.4f}")
print(f"\nNote: If all events already pass the trigger (≥3 b-tags),")
print(f"the ParT classifier provides discrimination *beyond* the trigger cut.")
print(f"\nPrecision-Recall AUC:")
prec, rec, _ = precision_recall_curve(labels, scores, sample_weight=qcd_w)
pr_auc = auc(rec, prec)
print(f"  PR-AUC = {pr_auc:.4f}")